# Gemma 4 TPU Fine-Tuning with TorchTPU & LiteRT-LM Deployment

<table class="tfo-notebook-buttons">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/colabs/Gemma4_E2B_TorchTPU_Fine_Tuning.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-ai-edge/LiteRT-LM/blob/main/colabs/Gemma4_E2B_TorchTPU_Fine_Tuning.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
    View source on GitHub</a>
  </td>
</table>

[LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM) is Google’s high-performance edge inference engine that runs LLMs across mobile, web, and desktop with full CPU/GPU/NPU acceleration and production-ready SDKs.

This notebook fine-tunes [Gemma 4 E2B](https://huggingface.co/google/gemma-4-E2B-it) or [Gemma 4 E4B](https://huggingface.co/google/gemma-4-E4B-it) on Cloud TPU using [TorchTPU + TorchTitan](https://github.com/google-pytorch/torchtitan/tree/main/torchtitan/experiments/tpu), then converts the model to `.litertlm` for on-device inference.

- **Part A — TPU Fine-Tuning**: Fine-tunes the text decoder on single-turn **GSM8K** or **QA** pairs while keeping vision/audio towers frozen.
- **Part B — LiteRT-LM Export & Verification**: Quantizes the merged checkpoint with Converter V2 (`use_v2=True`) and verifies inference in `litert_lm.Engine`.

**Setup**:
- Connect to either a **`v5e-1`** or **`v6e-1`** (or multi-chip `v6e-4` / `v6e-8`) TPU runtime (**Runtime $\rightarrow$ Change runtime type $\rightarrow$ TPU**).
- **Estimated Runtime**: Expect **~30 minutes** end-to-end on a `v5e-1` or `v6e-1` TPU with the default `TRAIN_STEPS = 30` (longer if you increase `TRAIN_STEPS`).
- Add `HF_TOKEN` in Colab **Secrets (🔑)** if downloading gated Gemma 4 repositories or uploading checkpoints to Hugging Face Hub.

## Part A — Fine-Tuning Gemma 4 on TPU (`TorchTPU` + `TorchTitan`)

### Step 1: Install `torch_tpu` + `torchtitan` & Verify TPU

In [ ]:
# @title { display-mode: "form" }
import glob
import importlib.metadata as md
import os
import subprocess
import sys

TORCHTITAN_DIR = "/content/torchtitan"

if not os.path.exists(TORCHTITAN_DIR):
    print("Cloning https://github.com/google-pytorch/torchtitan.git (main) ...")
    subprocess.run(
        ["git", "clone", "https://github.com/google-pytorch/torchtitan.git", TORCHTITAN_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", TORCHTITAN_DIR, "pull", "--ff-only"], check=False)

abi_pins = []
for pkg in ("numpy", "pyarrow", "pillow"):
    try:
        abi_pins.append(f"{pkg}=={md.version(pkg)}")
    except Exception:
        pass

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "keyrings.google-artifactregistry-auth"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.15.0", "torch_tpu==0.2.0",
        "-r", f"{TORCHTITAN_DIR}/torchtitan/experiments/tpu/requirements.txt",
        "--extra-index-url", "https://download.pytorch.org/whl/test/cpu",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{TORCHTITAN_DIR}/requirements.txt"],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"],
    check=False,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "--upgrade-strategy", "only-if-needed",
        "transformers>=5.19.0", "accelerate>=1.2.0",
        "huggingface_hub>=0.28.0", "safetensors>=0.4.5",
        *abi_pins,
    ],
    check=True,
)

vfio_devs = [p for p in glob.glob("/dev/vfio/[0-9]*") if os.path.basename(p).isdigit()]
accel_devs = glob.glob("/dev/accel[0-9]*")
detected_chips = len(vfio_devs) or len(accel_devs) or 1

# Verify TPU in a subprocess so the notebook kernel does not lock /dev/vfio/0 before torchrun
verify_cmd = (
    "import importlib.metadata as md, torch, torch_tpu; "
    f"cnt = torch.accelerator.device_count() if hasattr(torch, 'accelerator') else {detected_chips}; "
    "print('[OK] PyTorch=' + str(torch.__version__) + ' | torch_tpu=' + str(md.version('torch_tpu')) + f' | Visible TPU chips: {cnt}')"
)
res = subprocess.run([sys.executable, "-c", verify_cmd], capture_output=True, text=True)
print(res.stdout if res.returncode == 0 else f"[INFO] Detected TPU device nodes: {detected_chips}")

### Step 2: Stage Base Checkpoint & Prepare Single-Turn SFT Dataset

In [ ]:
# @title { display-mode: "form" }
import json
import os
from datasets import load_dataset
from huggingface_hub import login, snapshot_download

MODEL_CHOICE = "google/gemma-4-E2B-it"  # @param ["google/gemma-4-E2B-it", "google/gemma-4-E4B-it"]
DATASET_CHOICE = "gsm8k"  # @param ["gsm8k", "medqa"]
BASE_MODEL_DIR = "/content/base_hf_model"
SFT_DATA_DIR = "/content/sft_data"

os.makedirs(BASE_MODEL_DIR, exist_ok=True)
os.makedirs(SFT_DATA_DIR, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

if not os.path.exists(os.path.join(BASE_MODEL_DIR, "config.json")):
    print(f"Downloading {MODEL_CHOICE} to {BASE_MODEL_DIR} ...")
    snapshot_download(repo_id=MODEL_CHOICE, local_dir=BASE_MODEL_DIR, token=hf_token)
print(f"[OK] Base checkpoint staged at: {BASE_MODEL_DIR}")

if DATASET_CHOICE == "gsm8k":
    ds_train = load_dataset("openai/gsm8k", "main", split="train")
    ds_val = load_dataset("openai/gsm8k", "main", split="test[:256]")
    train_rows = [{"question": r["question"].strip(), "answer": r["answer"].strip()} for r in ds_train]
    val_rows = [{"question": r["question"].strip(), "answer": r["answer"].strip()} for r in ds_val]
else:
    ds_train = load_dataset("GBaker/MedQA-USMLE-4-options", split="train[:2000]")
    ds_val = load_dataset("GBaker/MedQA-USMLE-4-options", split="test[:256]")
    def _to_medqa(row):
        opts = row.get("options", {})
        opts_str = "\n".join(f"{k}: {v}" for k, v in sorted(opts.items()))
        q_text = (
            "You are a board-certified physician. Read the clinical vignette and "
            "respond with the correct option letter and answer.\n\n"
            f"Question:\n{row['question'].strip()}\n\nOptions:\n{opts_str}"
        )
        return {"question": q_text, "answer": f"The correct answer is {row.get('answer_idx', '').strip()}: {row.get('answer', '').strip()}"}
    train_rows = [_to_medqa(r) for r in ds_train]
    val_rows = [_to_medqa(r) for r in ds_val]

train_jsonl_path = os.path.join(SFT_DATA_DIR, "train.jsonl")
val_jsonl_path = os.path.join(SFT_DATA_DIR, "val.jsonl")

with open(train_jsonl_path, "w", encoding="utf-8") as f:
    for r in train_rows:
        f.write(json.dumps(r) + "\n")
with open(val_jsonl_path, "w", encoding="utf-8") as f:
    for r in val_rows:
        f.write(json.dumps(r) + "\n")

print(f"[OK] Wrote {DATASET_CHOICE} SFT JSONL splits: train={len(train_rows)} ({train_jsonl_path}), val={len(val_rows)} ({val_jsonl_path})")
print("Sample training pair:", json.dumps(train_rows[0], indent=2)[:400] + " ...")

### Step 3: Verify `hf_sft_full_max` SFT Config

In [ ]:
# @title { display-mode: "form" }
import os
import subprocess
import sys

TORCHTITAN_DIR = "/content/torchtitan"
tpu_exp_dir = os.path.join(TORCHTITAN_DIR, "torchtitan", "experiments", "tpu")

# TorchTitan moved TPU helpers into `distributed/` and `components/` subpackages; support both layouts.
for fname, old_cond, new_cond in [
    ("distributed_utils.py", "if num_devices <= 1:", "if num_devices < 1:"),
    ("cloud_distributed_utils.py", "if world_size == 1:", "if world_size < 1:"),
]:
    for fpath in (os.path.join(tpu_exp_dir, "distributed", fname), os.path.join(tpu_exp_dir, fname)):
        if os.path.isfile(fpath):
            with open(fpath, "r", encoding="utf-8") as f:
                src = f.read()
            if old_cond in src:
                with open(fpath, "w", encoding="utf-8") as f:
                    f.write(src.replace(old_cond, new_cond))
                print(f"[OK] Enabled single-chip (WORLD_SIZE=1) SliceBuilder init in `{os.path.relpath(fpath, tpu_exp_dir)}`")

check_code = f"""
import sys
sys.path.insert(0, {TORCHTITAN_DIR!r})
try:
    from torchtitan.experiments.tpu.components.datasets import CHAT_DATASETS
except ModuleNotFoundError:
    from torchtitan.experiments.tpu.datasets import CHAT_DATASETS
from torchtitan.experiments.tpu.transformers_modeling_backend.config_registry import hf_sft_full_max
cfg = hf_sft_full_max()
assert 'chat_json' in CHAT_DATASETS, 'Expected chat_json in CHAT_DATASETS'
model_spec = getattr(cfg, 'model_spec', None)
adapter = getattr(model_spec, 'state_dict_adapter', None)
model_cfg = getattr(model_spec, 'model', None)
print('[OK] Verified upstream `hf_sft_full_max` recipe & `chat_json` dataset loader in TorchTitan:')
print('  - state_dict_adapter : ' + str(getattr(adapter, '__name__', adapter)))
print('  - attn_mask_type     : ' + str(getattr(model_cfg, 'attn_mask_type', 'n/a')))
print('  - supported datasets : ' + str(sorted(CHAT_DATASETS.keys())))
"""
res = subprocess.run([sys.executable, "-c", check_code], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
else:
    err_tail = "\n".join((res.stderr or res.stdout).strip().splitlines()[-25:])
    raise RuntimeError(
        "TorchTitan `hf_sft_full_max` / `chat_json` check failed. Make sure Step 1 completed "
        "successfully (torch_tpu + torchtitan requirements installed).\n--- Error ---\n" + err_tail
    )

### Step 4: Launch Distributed TPU Fine-Tuning

Launches supervised fine-tuning in `bfloat16` precision on TPU via `torchrun`. The default `TRAIN_STEPS = 30` keeps the end-to-end run around 30 minutes; increase it for better quality at the cost of a longer run.

In [ ]:
# @title { display-mode: "form" }
import glob
import os
import socket
import subprocess
import sys

TRAIN_STEPS = 30  # @param {type:"integer"}
VAL_FREQ = 10  # @param {type:"integer"}
DUMP_FOLDER = "/content/torchtitan_outputs"

TORCHTITAN_DIR = "/content/torchtitan"
if TORCHTITAN_DIR not in sys.path:
    sys.path.insert(0, TORCHTITAN_DIR)

try:
    from torchtitan.experiments.tpu.distributed import local_tpu_device
except ImportError:
    from torchtitan.experiments.tpu import local_tpu_device

tpu_chip, detected_chips = local_tpu_device.get_local_chips()
vfio_devs = [p for p in glob.glob("/dev/vfio/[0-9]*") if os.path.basename(p).isdigit()]
accel_devs = glob.glob("/dev/accel[0-9]*")
nproc_per_node = detected_chips or len(vfio_devs) or len(accel_devs) or 1
tokens_per_microbatch = 2048 if nproc_per_node == 1 else 4096
ctx_len = 1024 if nproc_per_node == 1 else 2048

default_topo = {1: "1,1,1", 2: "1,2,1", 4: "2,2,1", 8: "2,4,1"}.get(nproc_per_node, "1,1,1")
chips_per_host_bounds = "1,1,1"

socks, sb_ports = [], []
for _ in range(nproc_per_node):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(("localhost", 0))
    sb_ports.append(s.getsockname()[1])
    socks.append(s)
for s in socks:
    s.close()

sb_addresses = ",".join(f"localhost:{p}" for p in sb_ports)
tpu_topology = os.environ.get("TORCH_TPU_TOPOLOGY") or default_topo

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc_per_node}",
    "-m",
    "torchtitan.experiments.tpu.train",
    "--module=torchtitan.experiments.tpu.transformers_modeling_backend",
    "--config=hf_sft_full_max",
    f"--dump_folder={DUMP_FOLDER}",
    f"--hf_model={BASE_MODEL_DIR}",
    f"--hf_assets_path={BASE_MODEL_DIR}",
    "--dataloader.dataset=chat_json",
    f"--dataloader.dataset_path={SFT_DATA_DIR}/train.jsonl",
    "--dataloader.prompt_field=question",
    "--dataloader.response_field=answer",
    "--validator.dataloader.dataset=chat_json",
    f"--validator.dataloader.dataset_path={SFT_DATA_DIR}/val.jsonl",
    "--validator.dataloader.prompt_field=question",
    "--validator.dataloader.response_field=answer",
    f"--validator.freq={VAL_FREQ}",
    "--validator.steps=5",
    "--lr_scheduler.warmup_steps=5",
    "--training.dtype=bfloat16",
    "--training.mixed_precision_param=bfloat16",
    "--training.mixed_precision_reduce=bfloat16",
    f"--training.steps={TRAIN_STEPS}",
    f"--training.max_context_length={ctx_len}",
    f"--training.num_tokens_per_microbatch_per_dp_rank={tokens_per_microbatch}",
    "--checkpoint.last_save_in_hf",
    "--checkpoint.export_dtype=bfloat16",
]

print(f"Launching TorchTitan TPU fine-tuning on {nproc_per_node} chip(s)...")
subprocess.run(["fuser", "-k", "/dev/vfio/0"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
if os.path.exists("/tmp/libtpu_lockfile"):
    os.remove("/tmp/libtpu_lockfile")

env = os.environ.copy()
env["PYTHONPATH"] = f"{TORCHTITAN_DIR}:{env.get('PYTHONPATH', '')}"
env["TORCH_TPU_SLICEBUILDER_ADDRESSES"] = sb_addresses
env["TORCH_TPU_TOPOLOGY"] = tpu_topology
env["TORCH_DYNAMO_AUTOMATIC_DYNAMIC_SHAPES"] = "0"
if nproc_per_node == 1:
    env["CLOUD_TPU_TASK_ID"] = "0"
    env["TPU_VISIBLE_CHIPS"] = "0"
    env["TPU_CHIPS_PER_HOST_BOUNDS"] = chips_per_host_bounds
    env["TPU_HOST_BOUNDS"] = tpu_topology
    env["TPU_PROCESS_ADDRESSES"] = sb_addresses
    env["TPU_PROCESS_PORT"] = str(sb_ports[0])

import collections
log_tail = collections.deque(maxlen=80)
proc = subprocess.Popen(cmd, cwd=TORCHTITAN_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    log_tail.append(line)
    if any(k in line for k in ("step:", "validate step:", "Total parameter count", "Saving the checkpoint", "Training completed")):
        print(line, end="")
if proc.wait() != 0:
    raise RuntimeError(
        "TorchTitan TPU training exited with non-zero status. Last log lines:\n" + "".join(log_tail)
    )
print(f"\n[OK] TorchTitan TPU fine-tuning finished! Output folder: {DUMP_FOLDER}")

### Step 5: Assemble Multimodal Hugging Face Checkpoint

Merges the fine-tuned text decoder weights with the frozen vision/audio towers into `/content/gemma4_tpu_finetuned_hf`. If uploading to Hugging Face Hub, set `HF_REPO_ID` and an individual `HF_SUBFOLDER` to avoid overwriting shared files.

In [ ]:
# @title { display-mode: "form" }
import glob
import os
import shutil
import subprocess
import sys

UPLOAD_HF_CHECKPOINT_TO_HUB = False  # @param {type:"boolean"}
HF_REPO_ID = "your-hf-username/gemma4-e2b-torchtpu-litertlm"  # @param {type:"string"}
HF_SUBFOLDER = "gemma4_tpu_finetuned_hf"  # @param {type:"string"}

FINAL_HF_CHECKPOINT_DIR = "/content/gemma4_tpu_finetuned_hf"
step_ckpt_dir = os.path.join(DUMP_FOLDER, "checkpoint", f"step-{TRAIN_STEPS}")

merge_script = f"""
import glob, os, shutil
from safetensors import safe_open
from safetensors.torch import save_file

step_ckpt_dir = {step_ckpt_dir!r}
base_model_dir = {BASE_MODEL_DIR!r}
output_dir = {FINAL_HF_CHECKPOINT_DIR!r}
os.makedirs(output_dir, exist_ok=True)

shard_files = sorted(glob.glob(os.path.join(step_ckpt_dir, "model-*.safetensors")))
finetuned_st_files = shard_files or sorted(glob.glob(os.path.join(step_ckpt_dir, "*.safetensors")))
if not finetuned_st_files:
    finetuned_st_files = sorted(glob.glob(os.path.join(step_ckpt_dir, "sharded", "*.safetensors")))
if not finetuned_st_files:
    raise FileNotFoundError(f"No .safetensors files found in {{step_ckpt_dir}}.")

print(f"Loading fine-tuned weights from {{finetuned_st_files}} ...")
finetuned_tensors = {{}}
for st_path in finetuned_st_files:
    with safe_open(st_path, framework="pt", device="cpu") as f:
        for k in f.keys():
            finetuned_tensors[k] = f.get_tensor(k)
print(f"[OK] Loaded {{len(finetuned_tensors)}} fine-tuned language-model tensors.")

for fname in os.listdir(base_model_dir):
    if fname.endswith(".safetensors") or fname == "model.safetensors.index.json":
        continue
    src_path = os.path.join(base_model_dir, fname)
    if os.path.isfile(src_path):
        shutil.copy2(src_path, os.path.join(step_ckpt_dir, fname))
        shutil.copy2(src_path, os.path.join(output_dir, fname))

base_st_files = sorted(glob.glob(os.path.join(base_model_dir, "*.safetensors")))
merged_tensors = {{}}
replaced_count, frozen_copied_count = 0, 0
for st_path in base_st_files:
    with safe_open(st_path, framework="pt", device="cpu") as f:
        for k in f.keys():
            if k in finetuned_tensors:
                merged_tensors[k] = finetuned_tensors[k]
                replaced_count += 1
            else:
                merged_tensors[k] = f.get_tensor(k)
                frozen_copied_count += 1

for k, v in finetuned_tensors.items():
    if k not in merged_tensors:
        merged_tensors[k] = v

out_safetensors_path = os.path.join(output_dir, "model.safetensors")
save_file(merged_tensors, out_safetensors_path, metadata={{"format": "pt"}})
shutil.copy2(out_safetensors_path, os.path.join(step_ckpt_dir, "model.safetensors"))

size_mb = os.path.getsize(out_safetensors_path) / (1024 * 1024)
print(f"[OK] Assembled HF checkpoint in {{step_ckpt_dir}} and {{output_dir}}:")
print(f"  - Fine-tuned language-model tensors : {{replaced_count}}")
print(f"  - Frozen tower tensors copied       : {{frozen_copied_count}}")
print(f"  - Total tensors                     : {{len(merged_tensors)}} ({{size_mb:.1f}} MB)")
"""
subprocess.run([sys.executable, "-c", merge_script], check=True)

if UPLOAD_HF_CHECKPOINT_TO_HUB and HF_REPO_ID and not HF_REPO_ID.startswith("your-hf-username/"):
    from huggingface_hub import HfApi
    api = HfApi(token=globals().get("hf_token"))
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    subfolder = HF_SUBFOLDER.strip("/") or "gemma4_tpu_finetuned_hf"
    api.upload_folder(
        folder_path=FINAL_HF_CHECKPOINT_DIR,
        path_in_repo=subfolder,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"[OK] Uploaded fine-tuned HF checkpoint to https://huggingface.co/{HF_REPO_ID}/tree/main/{subfolder}")

## Part B — Blockwise-Int4 PTQ & LiteRT-LM Deployment (`.litertlm`)

### Step 6: Export Fine-Tuned Model to `.litertlm` (Converter V2)

**Quantization Recipe Trade-offs (`QUANTIZATION_RECIPE`)**:
- **`dynamic_wi4b32_octav_afp32` (Default)**: 4-bit blockwise weight quantization (block size 32) using OCTAV optimal clipping with `float32` activations — balances compact size (~1.5 GB for E2B) and high reasoning accuracy.
- **`gemma4_mixed48_b32_octav`**: Mixed 4-bit/8-bit precision that keeps per-layer embeddings in 8-bit while quantizing transformer blocks to 4-bit OCTAV for higher accuracy with a slightly larger footprint.
- **`dynamic_wi4b32_afp32`**: Standard min-max 4-bit blockwise quantization without OCTAV clipping.

In [ ]:
# @title { display-mode: "form" }
import importlib.metadata as md
import os
import shutil
import subprocess
import sys

QUANTIZATION_RECIPE = "dynamic_wi4b32_octav_afp32"  # @param ["dynamic_wi4b32_octav_afp32", "gemma4_mixed48_b32_octav", "dynamic_wi4b32_afp32"]
CACHE_LENGTH = 2048  # @param {type:"integer"}
REFERENCE_HF_REPO_ID = "litert-community/Gemma4_E2B_Nepali_demo"

FINAL_HF_CHECKPOINT_DIR = globals().get("FINAL_HF_CHECKPOINT_DIR", "/content/gemma4_tpu_finetuned_hf")
EXPORT_DIR = "/content/litertlm_export"

if not os.path.isfile(os.path.join(FINAL_HF_CHECKPOINT_DIR, "model.safetensors")):
    from huggingface_hub import snapshot_download
    os.makedirs(FINAL_HF_CHECKPOINT_DIR, exist_ok=True)
    tmp_dl = snapshot_download(repo_id=REFERENCE_HF_REPO_ID, allow_patterns=["gemma4_tpu_finetuned_hf/*"])
    sub_dir = os.path.join(tmp_dl, "gemma4_tpu_finetuned_hf")
    for fname in os.listdir(sub_dir):
        shutil.copy2(os.path.join(sub_dir, fname), os.path.join(FINAL_HF_CHECKPOINT_DIR, fname))
    print(f"[OK] Restored fine-tuned HF checkpoint to {FINAL_HF_CHECKPOINT_DIR}")

abi_pins = []
for pkg in ("numpy", "pyarrow", "pillow"):
    try:
        abi_pins.append(f"{pkg}=={md.version(pkg)}")
    except Exception:
        pass

print("Installing litert-torch-nightly and litert-lm-nightly (Converter V2)...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "litert-torch", "litert-lm"], check=False)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "--upgrade-strategy", "only-if-needed",
        "tf-nightly", "litert-torch-nightly", "ai-edge-quantizer-nightly",
        "litert-lm-nightly", "litert-lm-builder", "litert-converter", "qrcode[pil]",
        *abi_pins,
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], check=False)

export_script = f"""
import logging, os, shutil
import numpy as np
from ai_edge_quantizer.algorithms.uniform_quantize import uniform_quantize_tensor as uqt
from litert_torch.generative.export_hf import export as export_hf
from litert_torch.generative.export_hf.core import quantization as export_quant_mod

logging.getLogger("transformers").setLevel(logging.CRITICAL)

orig_zp_scale_fn = uqt.tensor_zp_scale_from_min_max
def _safe_zp_scale_from_min_max(*args, **kwargs):
    zp, scale = orig_zp_scale_fn(*args, **kwargs)
    granularity = kwargs.get("granularity", args[4] if len(args) > 4 else None)
    if granularity is not None and uqt.is_blockwise(granularity):
        scale = np.maximum(scale, np.float32(6.58e-5))
    return zp, scale
uqt.tensor_zp_scale_from_min_max = _safe_zp_scale_from_min_max

rlib = export_quant_mod.recipe_lib
if hasattr(rlib, "AlgorithmName") and hasattr(rlib.AlgorithmName, "OCTAV"):
    def _dynamic_wi4b32_octav_afp32():
        return rlib.dynamic_wi4b32_afp32() + rlib.dynamic_wi4b32_afp32(
            algorithm_key=rlib.AlgorithmName.OCTAV,
            operation_name=rlib.TFLOperationName.FULLY_CONNECTED,
        )
    def _gemma4_mixed48_b32_octav():
        return (
            rlib.dynamic_wi4b32_afp32()
            + rlib.dynamic_wi4b32_afp32(
                algorithm_key=rlib.AlgorithmName.OCTAV,
                operation_name=rlib.TFLOperationName.FULLY_CONNECTED,
            )
            + rlib.dynamic_wi8c_afp32(
                algorithm_key=rlib.AlgorithmName.OCTAV,
                regex="per_layer",
                operation_name=rlib.TFLOperationName.FULLY_CONNECTED,
            )
        )
    for rname, rfn in (("dynamic_wi4b32_octav_afp32", _dynamic_wi4b32_octav_afp32), ("gemma4_mixed48_b32_octav", _gemma4_mixed48_b32_octav)):
        export_quant_mod._LOCAL_QUANTIZATION_RECIPES[rname] = rfn
        setattr(rlib, rname, rfn)

active_recipe = {QUANTIZATION_RECIPE!r} if {QUANTIZATION_RECIPE!r} in export_quant_mod._LOCAL_QUANTIZATION_RECIPES else "dynamic_wi4b32_afp32"
shutil.rmtree({EXPORT_DIR!r}, ignore_errors=True)
os.makedirs({EXPORT_DIR!r}, exist_ok=True)

export_hf.export(
    model={FINAL_HF_CHECKPOINT_DIR!r},
    output_dir={EXPORT_DIR!r},
    quantization_recipe=active_recipe,
    vision_encoder_quantization_recipe="dynamic_wi8c_afp32",
    audio_encoder_quantization_recipe="dynamic_wi8c_afp32",
    task="multimodal_lm",
    export_vision_encoder=True,
    export_audio_encoder=False,
    use_bool_mask=True,
    sliding_window_ring_buffer_size=512,
    apply_gpu_composites=True,
    use_sdpa_composite=True,
    use_rope_composite=True,
    llm_metadata_max_num_tokens_override={CACHE_LENGTH},
    cache_length={CACHE_LENGTH},
    prefill_lengths=[128, 1024],
    keep_temporary_files=False,
    gemma4_vision_max_soft_tokens=280,
    use_v2=True,
)
"""
proc = subprocess.Popen([sys.executable, "-c", export_script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError("Gemma 4 LiteRT-LM export failed.")

LITERTLM_MODEL_PATH = f"{EXPORT_DIR}/model.litertlm"

### Step 7: Verify On-Device Inference with `litert_lm.Engine`

In [ ]:
import litert_lm

cpu_backend = (
    litert_lm.Backend.CPU()
    if callable(getattr(litert_lm.Backend, "CPU", None))
    else litert_lm.Backend.CPU
)
test_prompt = val_rows[0]["question"] if "val_rows" in globals() and val_rows else "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
print(f"Prompt:\n{test_prompt}\n")

model_path = globals().get("LITERTLM_MODEL_PATH", "/content/litertlm_export/model.litertlm")
with litert_lm.Engine(model_path, backend=cpu_backend, max_num_tokens=1024) as engine:
    with engine.create_conversation() as conv:
        reply = conv.send_message(test_prompt)
        text = "".join(p.get("text", "") for p in reply.get("content", []) if isinstance(p, dict)) if isinstance(reply, dict) else str(reply)
        print("LiteRT-LM (.litertlm) Response:\n" + text)

### Step 8: Share `.litertlm` via Hugging Face Hub

Uploads the exported `.litertlm` package to an **individual directory** (`HF_EXPORT_SUBFOLDER`) inside your Hugging Face repository (`HF_EXPORT_REPO_ID`) so multiple runs or users never overwrite a shared file.

In [ ]:
# @title { display-mode: "form" }
import io
import os
import qrcode
from IPython.display import Image as IPyImage, display

UPLOAD_TO_HF_HUB = False  # @param {type:"boolean"}
HF_EXPORT_REPO_ID = "your-hf-username/gemma4-e2b-torchtpu-litertlm"  # @param {type:"string"}
HF_EXPORT_SUBFOLDER = "gemma4_torchtpu"  # @param {type:"string"}

quant_recipe = globals().get("QUANTIZATION_RECIPE", "dynamic_wi4b32_octav_afp32")
litertlm_path = globals().get("LITERTLM_MODEL_PATH", "/content/litertlm_export/model.litertlm")
litertlm_filename = f"gemma4_torchtpu_{quant_recipe}.litertlm"
subfolder = HF_EXPORT_SUBFOLDER.strip("/") or "gemma4_torchtpu"
path_in_repo = f"{subfolder}/{litertlm_filename}"

if UPLOAD_TO_HF_HUB:
    if not os.path.isfile(litertlm_path):
        raise FileNotFoundError(f"Cannot upload {litertlm_path}: file not found. Run Step 6 first.")
    if HF_EXPORT_REPO_ID.startswith("your-hf-username/"):
        raise ValueError("Please set HF_EXPORT_REPO_ID to your own Hugging Face repository.")
    from huggingface_hub import HfApi
    api = HfApi(token=globals().get("hf_token"))
    api.create_repo(repo_id=HF_EXPORT_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_file(
        path_or_fileobj=litertlm_path,
        path_in_repo=path_in_repo,
        repo_id=HF_EXPORT_REPO_ID,
        repo_type="model",
    )
    print(f"[OK] Uploaded {path_in_repo} to https://huggingface.co/{HF_EXPORT_REPO_ID}")

download_url = f"https://huggingface.co/{HF_EXPORT_REPO_ID}/resolve/main/{path_in_repo}"
print(f"Direct .litertlm URL: {download_url}")

qr_img = qrcode.make(download_url)
buf = io.BytesIO()
qr_img.save(buf, format="PNG")
display(IPyImage(data=buf.getvalue(), width=220))

## Next Steps: Run Your `.litertlm` Model Everywhere

Once you have exported your `.litertlm` model, you can deploy and test it across mobile, web, and desktop:

- **[LiteRT-LM CLI (`litert-lm`)](https://developers.google.com/edge/litert-lm/cli)**: Run, benchmark, and serve `.litertlm` models from your terminal (`pip install litert-lm && litert-lm run model.litertlm`).
- **[LiteRT-LM Web Demo](https://google-ai-edge.github.io/LiteRT-LM/web_demos/chat/index.html)**: Upload your `.litertlm` package in Chrome or Edge to run 100% client-side in the browser with WebGPU acceleration.
- **[Google AI Edge Gallery App](https://developers.google.com/edge/gallery)**: Import your `.litertlm` model on Android or iOS (via your Hugging Face download URL or QR code above) for interactive on-device inference.